In [ ]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    "brain_tumor_dataset/",
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    "brain_tumor_dataset/",
    target_size=(128,128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

/Users/gagan/untitled folder/brain-tumout-detection/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Found 204 images belonging to 2 classes.
Found 50 images belonging to 2 classes.


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(128,128,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # Freeze feature extractor

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

In [3]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [4]:

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5


2026-03-02 19:31:45.806471: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


7/7 [==============================] - 2s 98ms/step - loss: 0.7025 - accuracy: 0.6225 - val_loss: 0.4927 - val_accuracy: 0.7400
Epoch 2/5
7/7 [==============================] - 0s 51ms/step - loss: 0.4133 - accuracy: 0.8333 - val_loss: 0.4670 - val_accuracy: 0.7800
Epoch 3/5
7/7 [==============================] - 0s 61ms/step - loss: 0.3481 - accuracy: 0.8431 - val_loss: 0.3370 - val_accuracy: 0.8800
Epoch 4/5
7/7 [==============================] - 0s 46ms/step - loss: 0.2629 - accuracy: 0.9069 - val_loss: 0.3047 - val_accuracy: 0.8600
Epoch 5/5
7/7 [==============================] - 0s 46ms/step - loss: 0.2118 - accuracy: 0.9314 - val_loss: 0.2971 - val_accuracy: 0.8800


In [5]:
#unfreeze the base model for fine-tuning
base_model.trainable = True

# Freeze most layers, unfreeze last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [6]:
# recompile the model with a lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
7/7 [==============================] - 2s 102ms/step - loss: 0.5638 - accuracy: 0.7549 - val_loss: 0.2935 - val_accuracy: 0.8400
Epoch 2/5
7/7 [==============================] - 0s 57ms/step - loss: 0.4832 - accuracy: 0.7794 - val_loss: 0.2948 - val_accuracy: 0.8200
Epoch 3/5
7/7 [==============================] - 0s 57ms/step - loss: 0.3858 - accuracy: 0.8235 - val_loss: 0.3012 - val_accuracy: 0.8200
Epoch 4/5
7/7 [==============================] - 0s 57ms/step - loss: 0.3541 - accuracy: 0.8431 - val_loss: 0.3102 - val_accuracy: 0.8200
Epoch 5/5
7/7 [==============================] - 0s 56ms/step - loss: 0.3186 - accuracy: 0.8627 - val_loss: 0.3223 - val_accuracy: 0.8200
